[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/solutions/49_tensor_parallel_solution.ipynb)

# 🔴 Solution: Tensor Parallel MLP

Reference solution for Megatron-style column-parallel + row-parallel MLP, simulated with weight shards.

$$\text{MLP}(x) = \sum_{i=1}^{N} \text{GELU}(x W_1^{(i)}) W_2^{(i)}$$

In [ ]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass

In [ ]:
import torch
import torch.nn as nn

In [ ]:
# ✅ SOLUTION

class TensorParallelMLP(nn.Module):
    def __init__(self, d_model, d_ff, world_size):
        super().__init__()
        self.world_size = world_size
        shard_size = d_ff // world_size
        self.w1_shards = nn.ParameterList([
            nn.Parameter(torch.randn(d_model, shard_size) * (2 / d_model) ** 0.5)
            for _ in range(world_size)
        ])
        self.w2_shards = nn.ParameterList([
            nn.Parameter(torch.randn(shard_size, d_model) * (2 / d_ff) ** 0.5)
            for _ in range(world_size)
        ])

    def forward(self, x):
        output = None
        for w1, w2 in zip(self.w1_shards, self.w2_shards):
            h = torch.nn.functional.gelu(x @ w1)
            partial = h @ w2
            output = partial if output is None else output + partial
        return output

In [ ]:
# Verify: compare TensorParallelMLP against a standard single-GPU MLP
torch.manual_seed(42)
d_model, d_ff, world_size = 64, 256, 4
x = torch.randn(2, d_model)

tp_mlp = TensorParallelMLP(d_model, d_ff, world_size)
out = tp_mlp(x)
print("Output shape:", out.shape)  # expect (2, 64)

# Build equivalent full MLP from the sharded weights
w1_full = torch.cat([p.data for p in tp_mlp.w1_shards], dim=1)  # (d_model, d_ff)
w2_full = torch.cat([p.data for p in tp_mlp.w2_shards], dim=0)  # (d_ff, d_model)
ref = torch.nn.functional.gelu(x @ w1_full) @ w2_full

print("Max diff vs reference:", (out - ref).abs().max().item())  # expect ~0

In [ ]:
from torch_judge import check
check("tensor_parallel")